In [45]:
import pickle
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py
import os
from tqdm import tqdm

root_path = "./NormanWeissman2019_filtered.h5ad"
adata = ad.read_h5ad(root_path)

In [56]:
adata.obs

,guide_id,read_count,UMI_count,coverage,gemgroup,good_coverage,number_of_cells,tissue_type,cell_line,cancer,...,perturbation_type,celltype,organism,perturbation,nperts,ngenes,ncounts,percent_mito,percent_ribo,n_genes
TTGAACGAGACTCGGA,ARID1A_NegCtrl0;ARID1A_NegCtrl0,28684,1809,15.856274,2,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,ARID1A,1,3079,15097.0,5.815725,33.569583,3079
CGTTGGGGTGTTTGTG,BCORL1_NegCtrl0;BCORL1_NegCtrl0,18367,896,20.498884,7,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,BCORL1,1,2100,8551.0,4.104783,45.842592,2100
GAACCTAAGTGTTAGA,FOSB_NegCtrl0;FOSB_NegCtrl0,16296,664,24.542169,6,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,FOSB,1,2772,10999.0,5.655060,17.801618,2772
CCTTCCCTCCGTCATC,SET_KLF1;SET_KLF1,16262,850,19.131765,4,True,1,cell_line,K562,True,...,CRISPR,lymphoblasts,human,SET_KLF1,2,5385,38454.0,4.335050,38.165080,5385
TCAATCTGTCTTTCAT,OSR2_NegCtrl0;OSR2_NegCtrl0,16057,1067,15.048735,2,True,2,cell_line,K562,True,...,CRISPR,lymphoblasts,human,OSR2,1,4869,27926.0,5.084867,32.317554,4869
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGCGCAGTCATGCT,RHOXF2_NegCtrl0;RHOXF2_NegCtrl0,1,1,1.000000,2,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,RHOXF2,1,1853,5192.0,5.508475,31.798921,1853
TTTGCGCCAGGACCCT,BCL2L11_BAK1;BCL2L11_BAK1,1,1,1.000000,3,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,BCL2L11_BAK1,2,3508,15704.0,6.718034,38.334182,3508
TTTGCGCGTACTTGAC-1,CNN1_NegCtrl0;CNN1_NegCtrl0,1,1,1.000000,3,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,CNN1,1,3609,15054.0,5.633054,29.440680,3609
TTTGCGCTCTCGCATC-1,CEBPB_OSR2;CEBPB_OSR2,1,1,1.000000,6,False,0,cell_line,K562,True,...,CRISPR,lymphoblasts,human,CEBPB_OSR2,2,2576,6825.0,2.695971,16.879121,2576


In [46]:
# 2. 基本过滤（可按需调整）
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# 3. 保存原始 counts
adata.layers["counts"] = adata.X.copy()

# 4. 归一化 + log1p
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 5. 选前 2000 个 HVG
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat",   # 也可改成 "seurat_v3"
    subset=False
)

# 6. 提取 HVG
hvg_genes = adata.var_names[adata.var["highly_variable"]].tolist()

print(f"HVG 数量: {len(hvg_genes)}")
print(hvg_genes[:20])

# 7. 如需直接子集化
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
print(adata_hvg.shape)

HVG 数量: 2000
['RP11-34P13.8', 'SAMD11', 'ISG15', 'RNF223', 'TNFRSF4', 'ANKRD65', 'RP5-892K4.1', 'RP11-181G12.4', 'TNFRSF14', 'SMIM1', 'LINC00337', 'HES2', 'TNFRSF9', 'RP5-1115A15.1', 'CA6', 'GPR157', 'AL357140.1', 'PGD', 'RP4-635E18.9', 'DRAXIN']
(111445, 2000)


In [47]:
adata_hvg.var

,ensemble_id,ncounts,ncells,n_cells,highly_variable,means,dispersions,dispersions_norm
RP11-34P13.8,ENSG00000239945,10.0,10,10,True,0.000098,0.498784,2.194118
SAMD11,ENSG00000187634,2521.0,2210,2210,True,0.020147,0.504550,2.215926
ISG15,ENSG00000187608,70679.0,47823,47823,True,0.366669,0.214763,1.135671
RNF223,ENSG00000237330,1056.0,208,208,True,0.009527,2.407011,9.411290
TNFRSF4,ENSG00000186827,112.0,106,106,True,0.000800,0.158718,0.907941
...,...,...,...,...,...,...,...,...
MT-ATP6,ENSG00000198899,7457105.0,111420,111420,True,3.852468,1.551092,2.020403
MT-CO3,ENSG00000198938,21887124.0,111441,111441,True,4.926188,2.381223,1.222576
MT-ND4,ENSG00000198886,11478554.0,111418,111418,True,4.272743,1.841380,1.110049
MT-ND6,ENSG00000198695,539272.0,101559,101559,True,1.440172,0.731908,2.178184


In [48]:
adata_hvg.uns['genes_names'] = hvg_genes
adata_hvg.write_h5ad(f"./only_hvg/Norman_only_hvg.h5ad")


In [49]:
adata_hvg.uns
with open("var_dims.pkl", "wb") as f:
    pickle.dump({'gene_names':hvg_genes}, f)

In [50]:
adata.var["highly_variable"]

RP11-34P13.3    False
RP11-34P13.7    False
RP11-34P13.8     True
FO538757.3      False
FO538757.2      False
                ...  
AL592183.1      False
AC007325.4      False
AL354822.1      False
AC004556.1      False
AC240274.1      False
Name: highly_variable, Length: 22608, dtype: bool

In [51]:
adata_hvg.X.max()

8.545323

In [52]:
pert_list = adata_hvg.obs["perturbation"].unique().tolist()
len(pert_list)
print(pert_list[:10])

['ARID1A', 'BCORL1', 'FOSB', 'SET_KLF1', 'OSR2', 'KLF1_BAK1', 'FOXA3_FOXL2', 'TP73', 'HES7', 'IRF1_SET']


In [53]:
double_pert_list = [x for x in pert_list if '_' in x]
single_pert_list = [x for x in pert_list if '_' not in x]
print(f"single_len: {len(single_pert_list)}, double_len: {len(double_pert_list)}")

single_len: 106, double_len: 131


In [54]:
from sklearn.model_selection import train_test_split

# double_pert_list: 你的字符串列表
train_list, test_list = train_test_split(
    double_pert_list,
    test_size=0.1,
    random_state=42,
    shuffle=True,
)

print(test_list)

['MAP2K6_ELMSAN1', 'PTPN12_ZBTB25', 'FOXA3_FOXF1', 'KIF18B_KIF2C', 'POU3F2_CBFA2T3', 'CEBPE_SPI1', 'FOSB_PTPN12', 'MAP2K6_IKZF3', 'SGK1_S1PR2', 'KLF1_TGFBR2', 'FOXL2_HOXB9', 'ZBTB10_SNAI1', 'CEBPE_CNN1', 'IGDCC3_MAPK1']


In [55]:
for x in tqdm(double_pert_list):
    split_x = x.split("_")
    new_x = split_x[-1] + "_" + split_x[0]
    
    if new_x in double_pert_list:
        print("both exist")


100%|██████████| 131/131 [00:00<00:00, 705332.25it/s]
